## Loading the model (distilbert-base-uncased-finetuned-sst-2-english)


In [16]:
import torch
from transformers import DistilBertTokenizer, pipeline

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
model = pipeline(
    task='text-classification',
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Device set to use cuda:0


# Loading the data


In [17]:
import pandas as pd

df = pd.read_csv("../data/scored_data/cleaned_reviews_no_special_no_lowercase_majority.csv")

## Output prediction

In [18]:
# our mappings are 0 negative and 1 positive
def predict_sentiment(text):
    result = model.predict(text)[0]
    return 1 if result['label'] == 'POSITIVE' else 0

In [19]:
df["transformer_prediction"] = df.apply(
    lambda x: predict_sentiment(str(x["review_title"]) + " " + str(x["review_body"])), axis=1
)

stratified_sample = df.groupby('category', group_keys=False).apply(lambda x: x.sample(5), include_groups=False)
stratified_sample[["review_title", "review_body", "transformer_prediction"]]

,review_title,review_body,transformer_prediction
207,NaN,love everything little projector get great ima...,0
164,Great little projector,purchase projector see version social media. h...,0
155,Smart Great quality!!,honestly super surprise quality projector. App...,1
215,NaN,"great price, take chance. WOW. Super easy use ...",1
201,NaN,"Not like pictured, refund also not provide ref...",0
21,mean,Basic cook oil work anything require high smok...,0
8,NaN,Great value good deal. Great value good buy. l...,1
63,Good affordable cook oil,use buy another cornnoul brand keep raise pric...,1
85,NaN,flexible cap. whenever use oil leak drop come ...,0
111,Good,"oil go-to versatility flavor. use baking, fryi...",1


## Model vs SOTA Metrics (same test split)

This section compares:
- **Optimized model (Dept 1 output):** GaussianNB from Task 4 setup
- **SOTA model:** `distilbert-base-uncased-finetuned-sst-2-english`

Ground truth is `final_sentiment` (binary only: positive/negative), and both models are evaluated on the **same test records**.

In [20]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ---------------- PREPARE LABELS ----------------
df_eval = df[df["final_sentiment"].isin(["positive", "negative"])].copy().reset_index(drop=True)

# SOTA label mapping from pipeline output to Task labels
# POSITIVE -> positive, NEGATIVE -> negative
if "transformer_prediction" not in df_eval.columns:
    raise ValueError("Missing 'transformer_prediction'. Run the SOTA inference cell first.")

df_eval["sota_label"] = df_eval["transformer_prediction"].map({1: "positive", 0: "negative"})

# ---------------- LOAD FEATURES FOR OPTIMIZED MODEL ----------------
X_glove = pd.read_csv("../task3/text representation/glove_no_special_no_lowercase.csv")
X_glove = X_glove.select_dtypes(include=["number"]).reset_index(drop=True)

# Align lengths between labels and features
min_len = min(len(df_eval), len(X_glove))
df_eval = df_eval.iloc[:min_len].reset_index(drop=True)
X_glove = X_glove.iloc[:min_len].reset_index(drop=True)

y = df_eval["final_sentiment"]
meta = df_eval[["category", "sota_label"]]

# Same split recipe used in Task 4
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X_glove,
    y,
    meta,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ---------------- OPTIMIZED MODEL (Dept 1) ----------------
optimized_model = GaussianNB()
optimized_model.fit(X_train, y_train)

optimized_pred = pd.Series(optimized_model.predict(X_test), index=y_test.index)
sota_pred = pd.Series(meta_test["sota_label"].values, index=y_test.index)

# ---------------- METRICS ----------------
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "F1 (positive)": f1_score(y_true, y_pred, pos_label="positive", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Confusion Matrix": confusion_matrix(y_true, y_pred).tolist()
    }

overall_results = pd.DataFrame(
    {
        "Optimized Model": compute_metrics(y_test, optimized_pred),
        "SOTA Model": compute_metrics(y_test, sota_pred)
    }
)

overall_results

,Optimized Model,SOTA Model
Accuracy,0.589041,0.835616
Precision,0.8,0.96875
Recall,0.380952,0.738095
F1 (positive),0.516129,0.837838
F1 (macro),0.579493,0.835586
Confusion Matrix,"[[27, 4], [26, 16]]","[[30, 1], [11, 31]]"


In [21]:
# Per-category breakdown (required: Accuracy + macro-F1)
per_category_rows = []

for category_name in sorted(meta_test["category"].dropna().unique()):
    mask = meta_test["category"] == category_name

    y_true_cat = y_test[mask]
    opt_cat = optimized_pred[mask]
    sota_cat = sota_pred[mask]

    if len(y_true_cat) == 0:
        continue

    per_category_rows.append(
        {
            "Category": category_name,
            "Optimized Acc": accuracy_score(y_true_cat, opt_cat),
            "Optimized F1 (macro)": f1_score(y_true_cat, opt_cat, average="macro", zero_division=0),
            "SOTA Acc": accuracy_score(y_true_cat, sota_cat),
            "SOTA F1 (macro)": f1_score(y_true_cat, sota_cat, average="macro", zero_division=0),
            "Samples": len(y_true_cat)
        }
    )

per_category_results = pd.DataFrame(per_category_rows).sort_values("Category").reset_index(drop=True)
per_category_results

,Category,Optimized Acc,Optimized F1 (macro),SOTA Acc,SOTA F1 (macro),Samples
0,electronics,0.444444,0.390977,0.851852,0.846591,27
1,food,0.800000,0.733333,0.700000,0.670330,20
2,furniture,0.576923,0.435897,0.923077,0.909722,26
